# ✦ SALES SIGNAL
### From advertising spend to a smarter sales decision

**An end-to-end regression project | Python · EDA · Feature engineering · Model selection · Business interpretation**

> A good prediction is not just a number. It is a better question for the next marketing meeting: *where should the next unit of budget go, and what evidence supports that choice?*

---

### Project brief
Use historical advertising spend across **TV, Radio and Newspaper** to estimate sales. Clean the source, explore relationships, prepare features, compare regression models fairly, and translate the findings into practical marketing guidance.

**Scope honesty:** this file contains 200 observations and no date, customer segment, or platform field. It supports cross-sectional regression, not time-series forecasting or segment/platform-level claims. Sales units and currency are not specified in the source, so results stay in the dataset's original units.

## 01 · The decision before the model

**Business question:** given a proposed mix of TV, radio and newspaper spend, what sales level does the historical pattern suggest?

**What the model can do**
- Estimate sales for combinations of the three recorded channels.
- Compare candidate models on unseen observations.
- Show which input features carry the strongest predictive signal.

**What it cannot prove**
- That increasing a channel *causes* sales to rise.
- That the same relationship will hold in a new market or future period.
- Which customer segment or digital platform performs best.

In [ ]:
%matplotlib inline
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
DATA_PATH = Path("data/Advertising.csv")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

## 02 · Load, inspect and clean the data

We preserve the original file, remove only the exported row-index column (`Unnamed: 0`), validate the required fields, convert columns to numeric, and check missing values, duplicates and impossible negative advertising spends.

**Visualization note:** charts are displayed directly below their code cells with `plt.show()` using Matplotlib's inline backend. Open this notebook in Jupyter or VS Code and choose **Run All** to see every chart in sequence.


In [ ]:
raw = pd.read_csv(DATA_PATH)
print(f"Raw shape: {raw.shape}")
display(raw.head())

data = raw.drop(columns=["Unnamed: 0"], errors="ignore").copy()
required = ["TV", "Radio", "Newspaper", "Sales"]

missing_columns = sorted(set(required) - set(data.columns))
if missing_columns:
    raise ValueError(f"Required columns are missing: {missing_columns}")

data = data[required].copy()
for col in required:
    data[col] = pd.to_numeric(data[col], errors="coerce")

print("\\nMissing values:")
display(data.isna().sum().rename("missing_count").to_frame())
print(f"Duplicate rows: {data.duplicated().sum()}")
print(f"Negative advertising values: {(data[['TV','Radio','Newspaper']] < 0).sum().sum()}")

if data.isna().any().any():
    raise ValueError("Resolve missing/non-numeric values before modeling.")
if (data[["TV", "Radio", "Newspaper"]] < 0).any().any():
    raise ValueError("Negative advertising spend is invalid for this project.")

print("\\nCleaned data summary:")
display(data.describe().T.round(2))

### Cleaning decision
No missing values are present in this dataset, so no imputation is needed. The index-like column is metadata rather than a predictor. We do not delete rows simply because they are unusual: an extreme observation may be a genuine campaign and should be investigated before removal.

## 03 · Explore the story in the numbers

Start with distributions and pairwise relationships. Correlation is useful for describing association, but it is not evidence of causation.

In [ ]:
features = ["TV", "Radio", "Newspaper"]
target = "Sales"

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.flat, features + [target]):
    ax.hist(data[col], bins=18, edgecolor="white")
    ax.set_title(f"{col} distribution")
    ax.set_xlabel(col)
    ax.set_ylabel("Observations")
plt.suptitle("A quick portrait of the dataset", y=1.02, fontsize=15)
plt.tight_layout()
plt.show()

corr = data.corr(numeric_only=True)
display(corr.round(3))
plt.figure(figsize=(7, 5))
plt.imshow(corr, aspect="auto")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=30, ha="right")
plt.yticks(range(len(corr.index)), corr.index)
plt.colorbar(label="Correlation")
plt.title("Correlation map: advertising inputs and sales")
plt.tight_layout()
plt.show()

### What stands out
In this sample, TV has the strongest simple linear association with sales, radio follows, and newspaper is weaker. This is a descriptive signal, not a budget-allocation rule by itself: channels may interact, and correlation does not establish incremental return on investment.

## 04 · Prepare the learning setup

We use an 80/20 train-test split with a fixed random seed. The test set is held back for a final, once-only check. Candidate selection uses shuffled 5-fold cross-validation on the training split only. The 20% holdout remains untouched until final evaluation.

In [ ]:
X = data[features]
y = data[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print(f"Training rows: {len(X_train)} | Holdout rows: {len(X_test)}")

## 05 · Train several models, not just one

We compare a simple linear baseline with regularized linear, degree-2 polynomial, Random Forest and Gradient Boosting models. Scaling and polynomial expansion are placed inside a `Pipeline`, so transformations are fitted within each training fold and do not leak validation information.

In [ ]:
models = {
    "Linear Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LinearRegression())
    ]),
    "Ridge Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=1.0))
    ]),
    "Polynomial Regression (degree 2)": Pipeline([
        ("polynomial", PolynomialFeatures(degree=2, include_bias=False)),
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=1.0))
    ]),
    "Random Forest": RandomForestRegressor(
        n_estimators=500, min_samples_leaf=2, max_features=1.0,
        random_state=RANDOM_STATE, n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=200, learning_rate=0.03, max_depth=2,
        loss="huber", random_state=RANDOM_STATE
    )
}

results, fitted_models = [], {}
for name, model in models.items():
    model.fit(X_train, y_train)
    holdout_pred = model.predict(X_test)
    cv_mae = -cross_val_score(
        model, X_train, y_train, cv=cv, scoring="neg_mean_absolute_error"
    ).mean()
    results.append({
        "Model": name,
        "5-fold CV MAE": cv_mae,
        "Holdout MAE": mean_absolute_error(y_test, holdout_pred),
        "Holdout RMSE": mean_squared_error(y_test, holdout_pred) ** 0.5,
        "Holdout R²": r2_score(y_test, holdout_pred)
    })
    fitted_models[name] = model

leaderboard = pd.DataFrame(results).sort_values("5-fold CV MAE").reset_index(drop=True)
display(leaderboard.round(3))

In [ ]:
# Visual model comparison — lower MAE is better
plt.figure(figsize=(10, 5))
chart_data = leaderboard.sort_values("5-fold CV MAE", ascending=True)
plt.barh(chart_data["Model"], chart_data["5-fold CV MAE"])
plt.xlabel("5-fold Cross-Validation MAE")
plt.ylabel("Regression Model")
plt.title("Model Performance Comparison")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

### How to read the scorecard
- **MAE:** average absolute prediction error, in the target's original units. Lower is better.
- **RMSE:** penalizes larger misses more heavily. Lower is better.
- **R²:** fraction of holdout variation explained by the model. Closer to 1 is better, but it is not a measure of causal impact.

The best candidate is selected by cross-validation MAE, then assessed on the untouched holdout set.

In [ ]:
best_name = leaderboard.loc[0, "Model"]
best_model = fitted_models[best_name]
best_pred = best_model.predict(X_test)

mae = mean_absolute_error(y_test, best_pred)
rmse = mean_squared_error(y_test, best_pred) ** 0.5
r2 = r2_score(y_test, best_pred)

print(f"Selected model: {best_name}")
print(f"Holdout MAE : {mae:.3f}")
print(f"Holdout RMSE: {rmse:.3f}")
print(f"Holdout R²  : {r2:.3f}")

## 06 · Does the model miss in a systematic way?

A strong score is not enough. Inspect predicted versus actual sales and the residual distribution. Patterns in residuals can signal a model that consistently misses certain ranges.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(y_test, best_pred, alpha=0.8)
low = min(y_test.min(), best_pred.min()) - 1
high = max(y_test.max(), best_pred.max()) + 1
axes[0].plot([low, high], [low, high], linestyle="--")
axes[0].set(xlabel="Actual sales", ylabel="Predicted sales",
            title="Actual vs predicted")
residuals = y_test - best_pred
axes[1].hist(residuals, bins=14, edgecolor="white")
axes[1].axvline(0, linestyle="--")
axes[1].set(xlabel="Residual (actual − predicted)", ylabel="Count",
            title="Residual distribution")
plt.tight_layout()
plt.show()

## 07 · Explain the model with permutation importance

Permutation importance measures how much the holdout error worsens when one feature is shuffled. It is a model-specific measure of predictive reliance, not a causal effect or a percentage return on ad spend.

In [ ]:
perm = permutation_importance(
    best_model, X_test, y_test, n_repeats=30,
    random_state=RANDOM_STATE, scoring="neg_mean_absolute_error"
)
importance = pd.DataFrame({
    "Feature": features,
    "MAE increase when shuffled": perm.importances_mean,
    "Variation across shuffles": perm.importances_std
}).sort_values("MAE increase when shuffled", ascending=False)
display(importance.round(3))

plt.figure(figsize=(8, 4.5))
plt.bar(importance["Feature"], importance["MAE increase when shuffled"])
plt.ylabel("Increase in MAE after shuffling")
plt.title("Which inputs does this model rely on most?")
plt.tight_layout()
plt.show()

## 08 · Turn a prediction into a business conversation

Use the helper below to test a hypothetical media mix. Enter spend values in the same units and scale as the original data. Do not extrapolate far beyond the observed ranges: a model trained on historical examples has little evidence about radically different budgets.

In [ ]:
def estimate_sales(tv, radio, newspaper):
    scenario = pd.DataFrame([{
        "TV": tv, "Radio": radio, "Newspaper": newspaper
    }])
    if (scenario[features] < 0).any().any():
        raise ValueError("Spend values must be non-negative.")
    return float(best_model.predict(scenario)[0])

# Example scenario — edit these values to match a real planning discussion.
example = {"TV": 150, "Radio": 25, "Newspaper": 20}
print("Scenario:", example)
print(f"Estimated sales: {estimate_sales(**example):.2f} (dataset units)")

### Practical recommendations
1. **Treat TV as a strong predictive signal in this sample**, but validate incremental performance with controlled tests before moving budget.
2. **Do not dismiss radio.** Its relationship with sales is meaningful in the observed data; test channel combinations rather than judging each channel in isolation.
3. **Ask for better campaign data.** Add dates, actual spend and sales by period, campaign objective, geography, audience segment, platform, reach, frequency, promotions, pricing and seasonality.
4. **Use experiments for causal decisions.** A/B tests, geo-lift tests or carefully designed marketing-mix models are more appropriate for estimating what extra spend *causes*.
5. **Monitor drift.** Refit and validate on newer data as consumer behavior, media costs and campaign strategy change.

**Important:** this dataset has no dates, so we cannot identify seasonal patterns, COVID-era effects or forecast a calendar-based future. It also has no target-segment or platform columns, so those comparisons require new data.

## 09 · Save the results

The model, scorecard and predictions can be exported for reuse. Keep the random seed and data version with the model so results can be reproduced.

In [ ]:
OUTPUT_DIR.mkdir(exist_ok=True)
leaderboard.to_csv(OUTPUT_DIR / "model_comparison.csv", index=False)
pd.DataFrame({
    "Actual Sales": y_test.values,
    "Predicted Sales": best_pred,
    "Prediction Error": y_test.values - best_pred
}).to_csv(OUTPUT_DIR / "test_predictions.csv", index=False)
importance.to_csv(OUTPUT_DIR / "permutation_importance.csv", index=False)

import joblib
joblib.dump(best_model, OUTPUT_DIR / "sales_prediction_model.joblib")
print("Saved scorecard, holdout predictions, importance and trained model.")

---
### Final takeaway

**The point of this pipeline is not to promise sales. It is to make marketing decisions more evidence-led.** The model learns patterns from the 200 historical observations, is checked against unseen rows, and makes its limitations visible. The next step toward a genuinely forward-looking sales forecast is collecting time-stamped campaign and sales data.